In [1]:
from google.colab import drive
import os, shutil, pathlib

# Try to unmount in case it's half-mounted
try:
    drive.flush_and_unmount()
except Exception as e:
    print("Unmount info:", e)

# Ensure clean local mountpoint
if os.path.exists("/content/drive"):
    try:
        # Only remove if it's NOT a mount
        if not os.path.ismount("/content/drive"):
            shutil.rmtree("/content/drive")
    except Exception as e:
        print("Cleanup info:", e)
os.makedirs("/content/drive", exist_ok=True)

# Mount fresh
drive.mount("/content/drive", force_remount=True)

# Then set ROOT
ROOT = "/content/drive/MyDrive/Dissertation/Dataset"

!pip install MNE

Mounted at /content/drive


In [2]:
import os, pathlib, yaml

ROOT = "/content/drive/MyDrive/Dissertation/Dataset"   # <--- change if needed
SUBDIRS = ["data/raw","data/interim","data/features","data/reports","data/splits","src","env"]
for sd in SUBDIRS:
    os.makedirs(f"{ROOT}/{sd}", exist_ok=True)
for sd in SUBDIRS:
    p = pathlib.Path(ROOT)/sd
    print(("✓" if p.exists() else "✗"), p)

# Preprocessing config (create if missing)
cfg_path = f"{ROOT}/env/preprocessing.yaml"
if not pathlib.Path(cfg_path).exists():
    preproc_cfg = {
      "fs_target": 256,
      "l_freq": 1.0,
      "h_freq": 40.0,
      "notch": 50.0,
      "tmin": -0.2,
      "tmax": 3.0,
      "baseline": [-0.2, 0.0],
      "ica_n": 20,
      "reject_uv": 100.0
    }
    with open(cfg_path, "w") as f:
        yaml.safe_dump(preproc_cfg, f, sort_keys=False)
    print("Wrote:", cfg_path)
else:
    print("Using existing:", cfg_path)

with open(cfg_path,"r") as f:
    CFG = yaml.safe_load(f)

RAW_DIR     = f"{ROOT}/data/raw"
INTERIM_DIR = f"{ROOT}/data/interim"
REPORTS_DIR = f"{ROOT}/data/reports"
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)


# Update the already-loaded config and persist it
CFG["fs_target"] = 256
with open(f"{ROOT}/env/preprocessing.yaml", "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)
print("Updated fs_target ->", CFG["fs_target"])

# All 32 DEAP subjects (s01 ... s32)
SUBJECTS = [f"s{i:02d}" for i in range(1, 33)]
print(SUBJECTS)




✓ /content/drive/MyDrive/Dissertation/Dataset/data/raw
✓ /content/drive/MyDrive/Dissertation/Dataset/data/interim
✓ /content/drive/MyDrive/Dissertation/Dataset/data/features
✓ /content/drive/MyDrive/Dissertation/Dataset/data/reports
✓ /content/drive/MyDrive/Dissertation/Dataset/data/splits
✓ /content/drive/MyDrive/Dissertation/Dataset/src
✓ /content/drive/MyDrive/Dissertation/Dataset/env
Using existing: /content/drive/MyDrive/Dissertation/Dataset/env/preprocessing.yaml
Updated fs_target -> 256
['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32']


In [3]:
import os, pathlib, yaml
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Dissertation/Dataset")
SUBDIRS = ["data/raw","data/interim","data/features","data/reports","data/splits","src","env"]
for sd in SUBDIRS:
    os.makedirs(f"{ROOT}/{sd}", exist_ok=True)
for sd in SUBDIRS:
    p = pathlib.Path(ROOT)/sd
    print(("✓" if p.exists() else "✗"), p)

# Preprocessing config (create if missing)
cfg_path = f"{ROOT}/env/preprocessing.yaml"
if not pathlib.Path(cfg_path).exists():
    preproc_cfg = {
      "fs_target": 256,
      "l_freq": 1.0,
      "h_freq": 40.0,
      "notch": 50.0,
      "tmin": -0.2,
      "tmax": 3.0,
      "baseline": [-0.2, 0.0],
      "ica_n": 20,
      "reject_uv": 100.0
    }
    with open(cfg_path, "w") as f:
        yaml.safe_dump(preproc_cfg, f, sort_keys=False)
    print("Wrote:", cfg_path)
else:
    print("Using existing:", cfg_path)

with open(cfg_path,"r") as f:
    CFG = yaml.safe_load(f)

RAW_DIR     = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
REPORTS_DIR = ROOT / "data" / "reports"
FEATURES_DIR = ROOT / "data" / "features"

os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)


# Update the already-loaded config and persist it
CFG["fs_target"] = 256
with open(f"{ROOT}/env/preprocessing.yaml", "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)
print("Updated fs_target ->", CFG["fs_target"])

# All 32 DEAP subjects (s01 ... s32)
SUBJECTS = [f"s{i:02d}" for i in range(1, 33)]
print(SUBJECTS)




✓ /content/drive/MyDrive/Dissertation/Dataset/data/raw
✓ /content/drive/MyDrive/Dissertation/Dataset/data/interim
✓ /content/drive/MyDrive/Dissertation/Dataset/data/features
✓ /content/drive/MyDrive/Dissertation/Dataset/data/reports
✓ /content/drive/MyDrive/Dissertation/Dataset/data/splits
✓ /content/drive/MyDrive/Dissertation/Dataset/src
✓ /content/drive/MyDrive/Dissertation/Dataset/env
Using existing: /content/drive/MyDrive/Dissertation/Dataset/env/preprocessing.yaml
Updated fs_target -> 256
['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32']


In [4]:
import numpy as np
from pathlib import Path
import mne
import matplotlib.pyplot as plt

# --- Channel typing so non-EEG don't affect EEG processing/montage ---
NON_EEG_MAP = {
    "EXG1":"eog","EXG2":"eog","EXG3":"eog","EXG4":"eog",
    "EXG5":"emg","EXG6":"emg","EXG7":"emg","EXG8":"emg",
    "GSR1":"misc","GSR2":"misc","Erg1":"misc","Erg2":"misc",
    "Resp":"misc","Plet":"misc","Temp":"misc","Status":"stim"
}

# --- (optional legacy) pick code that occurs 40× ---
def pick_code_with_40(events_all):
    if events_all is None or len(events_all)==0:
        return None
    ids, counts = np.unique(events_all[:,2], return_counts=True)
    hits = ids[counts == 40]
    if len(hits) == 1:
        return events_all[events_all[:,2] == hits[0]]
    return None

# --- Synthesize DEAP-like onsets (code fixed to 4) ---
def synthesize_deap_events_code4(raw, n_trials=40, baseline_s=3, clip_s=60):
    fs = raw.info["sfreq"]
    onsets = ((baseline_s + (baseline_s+clip_s)*np.arange(n_trials)) * fs).astype(int)
    return np.column_stack([onsets, np.zeros(n_trials, int), np.full(n_trials, 4, int)])

# --- Find Status==4 rising edges; keep ONE per trial via min-gap de-dup ---
def find_status_events_code4_one_per_trial(raw, expect_n=40, min_gap_sec=55):
    """
    Detect Status==4 events on ORIGINAL raw (do NOT filter Status).
    Keeps only rising edges and enforces a minimum gap between trials.
    """
    ev = mne.find_events(
        raw,
        stim_channel="Status",
        consecutive="increasing",   # rising edges only
        shortest_event=1,
        mask=4, mask_type='and',    # bit 3 set (value 4)
        verbose=False
    )
    if ev is None or len(ev)==0:
        return None
    # Keep strictly code==4 (safety)
    ev = ev[ev[:,2] == 4]
    # De-duplicate by enforcing a large inter-onset gap (~trial spacing in DEAP)
    fs = raw.info["sfreq"]
    min_gap = int(min_gap_sec * fs)     # try 55–65 s if needed
    keep = [0]
    for i in range(1, len(ev)):
        if ev[i,0] - ev[keep[-1],0] >= min_gap:
            keep.append(i)
    ev = ev[keep]
    if len(ev) != expect_n:
        print(f"[warn] code-4 events after de-dup: {len(ev)} (expected {expect_n})")
    return ev

# --- If you detected events at fs_src but will epoch at fs_dst, resample samples ---
def resample_events(events, fs_src, fs_dst):
    if fs_src == fs_dst:
        return events
    ev = events.copy()
    ev[:,0] = np.round(ev[:,0] * (fs_dst / fs_src)).astype(int)
    return ev

# --- Convenience: get code-4 events from raw_src; resample for raw_dst; fallback synth ---
def events_code4_or_synth(raw_src, raw_dst, expect_n=40, min_gap_sec=55):
    ev_raw = find_status_events_code4_one_per_trial(raw_src, expect_n=expect_n, min_gap_sec=min_gap_sec)
    if ev_raw is None:
        print("[info] No clean code-4 events; synthesizing (code=4).")
        return synthesize_deap_events_code4(raw_dst)
    return resample_events(ev_raw, raw_src.info["sfreq"], raw_dst.info["sfreq"])

# --- PSD helper using modern API ---
def raw_psd_welch(raw, fmin, fmax):
    """Welch PSD via the new API. Returns (psd, freqs)."""
    spec = raw.copy().pick_types(eeg=True).compute_psd(
        method="welch", fmin=fmin, fmax=fmax,
        n_fft=int(2*raw.info["sfreq"]), n_overlap=0, verbose=False
    )
    return spec.get_data(return_freqs=True)

# --- Keep only well-separated code-4 events (≥60 s gap); cap at 40 ---
def dedup_to_40(ev, fs, min_gap_sec=60):
    # ev: (n,3) at the *current* sampling rate
    ev = ev[ev[:,2] == 4]
    ev = ev[np.argsort(ev[:,0])]           # sort by onset
    keep = []
    min_gap = int(min_gap_sec * fs)
    for i in range(len(ev)):
        if not keep:
            keep.append(i)
        else:
            if ev[i,0] - ev[keep[-1],0] >= min_gap:
                keep.append(i)
    ev = ev[keep]
    return ev[:40]                          # enforce exactly 40 when possible

# Make MNE viewers render in notebooks/Colab
mne.viz.set_browser_backend("matplotlib")


Using matplotlib as 2D backend.


In [5]:
#Fixing status channel error
def fix_status_blank(subj: str):
    bdf_file = RAW_DIR / f"{subj}.bdf"
    if not bdf_file.exists():
        print(f"[ERROR] BDF file missing for {subj}: {bdf_file}")
        return
    raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
    if "Status" in raw.ch_names:
        print(f"{subj}: 'Status' already present. Skipping.")
        return
    blanks = [ch for ch in raw.ch_names if ch.strip() == ""]
    if len(blanks) != 1:
        print(f"{subj}: expected 1 blank channel, found {blanks}. Skipping.")
        return
    raw.rename_channels({blanks[0]: "Status"})
    raw.set_channel_types({"Status": "stim"})
    out_fif = RAW_DIR / f"{subj}_fixed_raw.fif"
    raw.save(str(out_fif), overwrite=True)
    print(f"{subj}: saved fixed FIF:", out_fif)

def detect_and_fix_status_minus(subj: str, expected_min_events=30):
    bdf_file = RAW_DIR / f"{subj}.bdf"
    if not bdf_file.exists():
        print(f"[ERROR] BDF file missing for {subj}: {bdf_file}")
        return

    raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)

    # If Status is already present, skip
    if "Status" in raw.ch_names:
        print(f"{subj}: 'Status' already present. Skipping.")
        return

    # Look for -0 / -1 candidates
    candidates = [ch for ch in raw.ch_names if ch in ("-0", "-1")]
    if not candidates:
        print(f"{subj}: no -0/-1 channels. Skipping.")
        return

    # Choose best candidate based on number of events
    best_ch, best_n = None, -1
    for ch in candidates:
        tmp = raw.copy()
        tmp.set_channel_types({ch: "stim"})
        try:
            ev = mne.find_events(tmp, stim_channel=ch,
                                 shortest_event=1,
                                 consecutive="increasing",
                                 verbose=False)
        except Exception as e:
            print(f"{subj}: find_events failed on {ch}: {e}")
            continue
        n_ev = len(ev)
        print(f"{subj}: candidate {ch} -> {n_ev} events")
        if n_ev > best_n:
            best_n = n_ev
            best_ch = ch

    if best_ch is None or best_n < expected_min_events:
        print(f"{subj}: couldn't identify Status. Best={best_ch}, n={best_n}")
        return

    # Rename best candidate to Status
    raw.rename_channels({best_ch: "Status"})
    raw.set_channel_types({"Status": "stim"})

    # DROP the leftover minus channel(s), if any
    for ch in ("-0", "-1"):
        if ch in raw.ch_names and ch != "Status":
            print(f"{subj}: dropping leftover channel {ch}")
            raw.drop_channels([ch])

    print("New channels:", raw.ch_names)

    out_fif = RAW_DIR / f"{subj}_fixed_raw.fif"
    raw.save(str(out_fif), overwrite=True)
    print(f"{subj}: saved fixed FIF:", out_fif)


# Run fixes
#for subj in [f"s{i:02d}" for i in range(24, 29)]:
 #   fix_status_blank(subj)

for subj in [f"s{i:02d}" for i in range(29, 33)]:
    detect_and_fix_status_minus(subj)


/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)


s29: candidate -0 -> 295 events
s29: candidate -1 -> 295 events
s29: dropping leftover channel -1
New channels: ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', 'Status']
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s29_fixed_raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s29_fixed_raw.fif
[done]
s29: saved fixed FIF: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s29_fixed_raw.fif


/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)


s30: candidate -0 -> 301 events
s30: candidate -1 -> 301 events
s30: dropping leftover channel -1
New channels: ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', 'Status']
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s30_fixed_raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s30_fixed_raw.fif
[done]
s30: saved fixed FIF: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s30_fixed_raw.fif


/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)


s31: candidate -0 -> 299 events
s31: candidate -1 -> 299 events
s31: dropping leftover channel -1
New channels: ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', 'Status']
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s31_fixed_raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s31_fixed_raw.fif
[done]
s31: saved fixed FIF: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s31_fixed_raw.fif


/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channel names are not unique, found duplicates for: {''}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)
/tmp/ipython-input-9252/2324389933.py:27: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(str(bdf_file), preload=True, verbose=False)


s32: candidate -0 -> 295 events
s32: candidate -1 -> 295 events
s32: dropping leftover channel -1
New channels: ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp', 'Status']
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s32_fixed_raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/raw/s32_fixed_raw.fif
[done]
s32: saved fixed FIF: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s32_fixed_raw.fif


In [6]:
from pathlib import Path
import numpy as np
import mne
import matplotlib.pyplot as plt

def load_and_clean_bdf(raw_path, cfg, do_ica=True, save_qc=True, reports_dir=None):
    """
    raw_path: Path or str to either a .bdf or .fif file.

    Returns:
      epochs        -> epoched, cleaned EEG (3 s)
      raw_after     -> continuous AFTER preprocessing
      events        -> event array used for epoching
      raw_before_eeg-> continuous BEFORE (EEG only)
    """
    raw_path = Path(raw_path)
    # Use subject id without suffix like "_fixed_raw"
    subj_id = raw_path.stem.split("_")[0]
    reports_dir = Path(reports_dir) if reports_dir is not None else Path(".")

    # --- Load raw (BDF or FIF) ---
    ext = raw_path.suffix.lower()
    if ext == ".bdf":
        raw = mne.io.read_raw_bdf(str(raw_path), preload=True, verbose=False)
    elif ext == ".fif":
        raw = mne.io.read_raw_fif(str(raw_path), preload=True, verbose=False)
    else:
        raise RuntimeError(f"Unsupported extension for {raw_path}")

    # --- Type non-EEG channels; montage ---
    mapping = {ch: tp for ch, tp in NON_EEG_MAP.items() if ch in raw.ch_names}
    if mapping:
        raw.set_channel_types(mapping)
    raw.set_montage("standard_1020", match_case=False, on_missing="ignore")

    # BEFORE (EEG only)
    raw_before_eeg = raw.copy().pick_types(eeg=True)

    # --- Preprocessing on a working copy ---
    raw_after = raw.copy()
    raw_after.set_eeg_reference("average")
    raw_after.filter(
        cfg["l_freq"], cfg["h_freq"],
        picks="eeg", fir_design="firwin", verbose=False
    )
    raw_after.notch_filter(cfg["notch"], picks="eeg", verbose=False)
    if cfg.get("fs_target"):
        raw_after.resample(cfg["fs_target"], npad="auto")

    # --- ICA (optional) ---
    if do_ica:
        ica = mne.preprocessing.ICA(
            n_components=cfg["ica_n"],
            random_state=97,
            max_iter="auto"
        )
        ica.fit(raw_after.copy().filter(1., None), verbose=False)
        try:
            eog_inds, _ = ica.find_bads_eog(raw_after)
            ica.exclude = eog_inds
        except Exception:
            pass
        raw_after = ica.apply(raw_after.copy())

    # --- events: detect from Status or synthesize; dedup to 40 ---
    events = events_code4_or_synth(
        raw_src=raw,
        raw_dst=raw_after,
        expect_n=40,
        min_gap_sec=60
    )
    events = dedup_to_40(events, fs=raw_after.info["sfreq"], min_gap_sec=60)
    print(
        f"{subj_id}: events used (after dedup):",
        events.shape, "unique codes:", np.unique(events[:, 2])
    )

    # save events at AFTER fs
    out_ev = INTERIM_DIR / f"{subj_id}-events-code4.npy"
    np.save(out_ev, events)
    print("Saved events to:", out_ev)

    # --- 3 s epochs for ERP QC (no rejection now to avoid all-dropped cases) ---
    reject = None  # IMPORTANT: avoid dropping everything on noisy subjects
    epochs = mne.Epochs(
        raw_after, events, event_id=4,
        tmin=cfg["tmin"], tmax=cfg["tmax"],
        baseline=tuple(cfg["baseline"]),
        picks="eeg",
        preload=True,
        reject=reject,
        verbose=False,
    )
    print(f"{subj_id}: short epochs (3 s) -> {len(epochs)}")

    # --- QC plots ---
    if save_qc:
        # PSD before/after
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        raw_before_eeg.plot_psd(fmin=1, fmax=40, axes=axes[0], show=False)
        axes[0].set_title("PSD BEFORE (1–40)")
        raw_after.copy().pick_types(eeg=True).plot_psd(
            fmin=1, fmax=40, axes=axes[1], show=False
        )
        axes[1].set_title("PSD AFTER (1–40)")
        fig.savefig(reports_dir / f"{subj_id}_psd.png", dpi=150, bbox_inches="tight")
        plt.close(fig)

        # ERP plot only if we actually have epochs
        if len(epochs) > 0:
            try:
                fig = epochs.copy().pick("Cz").average().plot(
                    spatial_colors=True, show=False
                )
            except Exception:
                fig = epochs.average().plot(spatial_colors=True, show=False)
            fig.savefig(reports_dir / f"{subj_id}_erp.png", dpi=150, bbox_inches="tight")
            plt.close(fig)
        else:
            print(f"{subj_id}: no short epochs; skipping ERP QC plot.")

    # ALWAYS return these four objects
    return epochs, raw_after, events, raw_before_eeg


In [7]:
import numpy as np
import mne
from pathlib import Path

def preprocess_subject(subj, cfg=CFG):
    """
    Run full preprocessing for one subject and save FIFs/epochs.
    Uses *_fixed_raw.fif if present, otherwise falls back to .bdf.
    """
    raw_bdf = RAW_DIR / f"{subj}.bdf"
    raw_fif = RAW_DIR / f"{subj}_fixed_raw.fif"

    # 0) choose input file
    if raw_fif.exists():
        raw_path = raw_fif
        print(f"{subj}: using FIXED raw file: {raw_fif}")
    elif raw_bdf.exists():
        raw_path = raw_bdf
        print(f"{subj}: using original BDF file: {raw_bdf}")
    else:
        print(f"[WARN] no raw file for {subj} (neither {raw_bdf} nor {raw_fif}). Skipping.")
        return None

    # 1) run cleaning
    epochs, raw_after, events, raw_before = load_and_clean_bdf(
        raw_path,
        cfg,
        do_ica=True,
        save_qc=True,
        reports_dir=REPORTS_DIR,
    )

    # 2) basic summary
    n_eeg = len(mne.pick_types(raw_after.info, eeg=True, exclude=[]))
    print(f"{subj}: EEG ch = {n_eeg}")
    print(f"{subj}: sfreq  = {raw_after.info['sfreq']}")
    print(f"{subj}: events = {events.shape[0]}, unique codes = {np.unique(events[:, 2])}")
    print(f"{subj}: epochs kept = {len(epochs)}/{events.shape[0]}")

    # 3) save continuous BEFORE/AFTER and short epochs (3 s)
    before_path = INTERIM_DIR / f"{subj}-raw.fif"
    after_path  = INTERIM_DIR / f"{subj}-clean-raw.fif"
    epo_path    = INTERIM_DIR / f"{subj}-epo.fif"

    raw_before.save(str(before_path), overwrite=True)
    raw_after.save(str(after_path), overwrite=True)
    epochs.save(str(epo_path), overwrite=True)

    # 4) build long 60s epochs used for features
    epochs60 = mne.Epochs(
        raw_after,
        events,
        event_id=4,
        tmin=0.0,
        tmax=60.0,
        baseline=None,
        picks="eeg",
        preload=True,
        reject=None,
        verbose=False,
    )
    epo60_path = INTERIM_DIR / f"{subj}-epo60.fif"
    epochs60.save(str(epo60_path), overwrite=True)

    print(
        f"{subj}: saved\n"
        f"  {before_path}\n"
        f"  {after_path}\n"
        f"  {epo_path}\n"
        f"  {epo60_path}"
    )

    return {
        "epochs": epochs,
        "epochs60": epochs60,
        "raw_after": raw_after,
        "raw_before": raw_before,
        "events": events,
    }


In [ ]:
SUBJECTS = [f"s{i:02d}" for i in range(1, 33)]
print(SUBJECTS)

for subj in SUBJECTS:
    print("=" * 60)
    print(f"Processing {subj}")
    _ = preprocess_subject(subj, CFG)


['s01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32']
Processing s01
s01: using original BDF file: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s01.bdf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipython-input-9252/1221179011.py:33: RuntimeWarning: The unit for channel(s) Erg1, Erg2, GSR1, GSR2, Plet, Resp, Temp has changed from V to NA.
  raw.set_channel_types(mapping)
/tmp/ipython-input-9252/1221179011.py:33: RuntimeWarning: The unit for channel(s) Status has changed from NA to V.
  raw.set_channel_types(mapping)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
13760 events found on stim channel Status
Event IDs: [1 2 3 4 5 6 7]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
11519 events found on stim channel Status
Event IDs: [1 2 3 4 5 6 7]
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transitio

/tmp/ipython-input-9252/1221179011.py:48: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  raw_after.resample(cfg["fs_target"], npad="auto")


Using EOG channels: EXG1, EXG2, EXG3, EXG4
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2560 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff fre

...

Plotting power spectral density (dB=True).
Need more than one channel to make topography for eeg. Disabling interactivity.
s20: EEG ch = 32
s20: sfreq  = 256.0
s20: events = 40, unique codes = [4]
s20: epochs kept = 40/40
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-raw.fif
[done]
Overwriting existing file.
Writing /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-clean-raw.fif
Overwriting existing file.
Closing /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-clean-raw.fif
[done]
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


/tmp/ipython-input-9252/4070776378.py:63: RuntimeWarning: This filename (/content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-epo60.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs60.save(str(epo60_path), overwrite=True)


s20: saved
  /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-raw.fif
  /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-clean-raw.fif
  /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-epo.fif
  /content/drive/MyDrive/Dissertation/Dataset/data/interim/s20-epo60.fif
Processing s21
s21: using original BDF file: /content/drive/MyDrive/Dissertation/Dataset/data/raw/s21.bdf


/tmp/ipython-input-9252/1221179011.py:33: RuntimeWarning: The unit for channel(s) Erg1, Erg2, GSR1, GSR2, Plet, Resp, Temp has changed from V to NA.
  raw.set_channel_types(mapping)
/tmp/ipython-input-9252/1221179011.py:33: RuntimeWarning: The unit for channel(s) Status has changed from NA to V.
  raw.set_channel_types(mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
289 events found on stim channel Status
Event IDs: [1 2 3 4 5 7]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
289 events found on stim channel Status
Event IDs: [1 2 3 4 5 7]
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 

In [ ]:
def compare_raw_vs_clean(root, subj, channel="Cz", t_start=120.0, win=5.0,
                         f_psd=(1,40), f_notch=(47,53), save_png=True):
    import numpy as np, matplotlib.pyplot as plt, pandas as pd, mne
    from pathlib import Path

    INTERIM = f"{root}/data/interim"
    p_raw   = Path(INTERIM)/f"{subj}-raw.fif"         # BEFORE
    p_clean = Path(INTERIM)/f"{subj}-clean-raw.fif"   # AFTER
    assert p_raw.exists() and p_clean.exists(), "Missing saved FIF files."

    raw_before = mne.io.read_raw_fif(str(p_raw), preload=True, verbose=False)
    raw_after  = mne.io.read_raw_fif(str(p_clean), preload=True, verbose=False)

    # Make sampling rates match
    if raw_before.info["sfreq"] != raw_after.info["sfreq"]:
        raw_before.resample(raw_after.info["sfreq"], npad="auto")
    fs = raw_after.info["sfreq"]

    # ---------- A) Time-domain overlay ----------
    if channel not in raw_before.ch_names:
        eeg_picks = mne.pick_types(raw_before.info, eeg=True, exclude=[])
        channel = raw_before.ch_names[eeg_picks[0]]
        print(f"Requested channel not found. Falling back to '{channel}'.")

    i0, i1 = int(t_start * fs), int((t_start + win) * fs)
    x_b = raw_before.copy().pick(channel).get_data()[0, i0:i1] * 1e6
    x_a = raw_after.copy().pick(channel).get_data()[0,  i0:i1] * 1e6
    x_b -= np.median(x_b); x_a -= np.median(x_a)
    t = np.arange(i1 - i0) / fs
    plt.figure(figsize=(10,4))
    plt.plot(t, x_b, label="BEFORE", alpha=0.7)
    plt.plot(t, x_a, label="AFTER",  alpha=0.9)
    plt.ylim(-20, 20)
    plt.xlabel("Time (s)"); plt.ylabel("Amplitude (µV)")
    plt.title(f"{subj} – {channel} – {win}s at t={t_start}s"); plt.legend(); plt.tight_layout()
    if save_png:
        out = Path(root)/"data/reports"/f"{subj}_{channel}_time_overlay.png"
        plt.savefig(out, dpi=150); print("Saved:", out)
    plt.show()

    # ---------- B) PSD plots ----------
    fmin, fmax = f_psd
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    raw_before.copy().pick_types(eeg=True).plot_psd(fmin=fmin, fmax=fmax, ax=axes[0], show=False)
    axes[0].set_title(f"PSD BEFORE ({fmin}-{fmax} Hz)")
    raw_after.copy().pick_types(eeg=True).plot_psd(fmin=fmin, fmax=fmax, ax=axes[1], show=False)
    axes[1].set_title(f"PSD AFTER ({fmin}-{fmax} Hz)")
    plt.tight_layout(); plt.show()

    f1, f2 = f_notch
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    raw_before.copy().pick_types(eeg=True).plot_psd(fmin=f1, fmax=f2, ax=axes[0], show=False)
    axes[0].set_title(f"PSD BEFORE ({f1}-{f2} Hz)")
    raw_after.copy().pick_types(eeg=True).plot_psd(fmin=f1, fmax=f2, ax=axes[1], show=False)
    axes[1].set_title(f"PSD AFTER ({f1}-{f2} Hz)")
    plt.tight_layout(); plt.show()

    # ---------- C) ERP/long-window overlay with 60 s epochs ----------
    LONG = True            # set True for 60 s analysis; False for 3 s ERP
    win_len = 60.0 if LONG else 3.0

    # use saved events at AFTER fs (your loader saved them)
    from pathlib import Path
    ev_path = Path(root)/"data"/"interim"/f"{subj}-events-code4.npy"
    if ev_path.exists():
        events_b = np.load(ev_path)
        events_b = dedup_to_40(events_b, fs=raw_after.info["sfreq"], min_gap_sec=60)
        events_a = events_b.copy()

    else:
      # fallback: detect from fixed FIF if exists, otherwise BDF, otherwise synthesize
      raw_dir = Path(root) / "data" / "raw"
      raw_fif = raw_dir / f"{subj}_fixed_raw.fif"
      raw_bdf = raw_dir / f"{subj}.bdf"

      if raw_fif.exists():
          raw_src = mne.io.read_raw_fif(str(raw_fif), preload=False, verbose=False)
          print(f"{subj}: compare_raw_vs_clean using FIXED raw for event detection:", raw_fif)
      elif raw_bdf.exists():
          raw_src = mne.io.read_raw_bdf(str(raw_bdf), preload=False, verbose=False)
          print(f"{subj}: compare_raw_vs_clean using original BDF for event detection:", raw_bdf)
      else:
          print(f"{subj}: no raw file found for event detection, synthesizing events.")
          events_b = synthesize_deap_events_code4(raw_before)
          events_a = synthesize_deap_events_code4(raw_after)
          goto_events_done = True

      if 'goto_events_done' not in locals():
          try:
              ev_raw = find_status_events_code4_one_per_trial(raw_src, expect_n=40, min_gap_sec=60)
          except ValueError as e:
              # Stim channel missing, synthesize
              print(f"{subj}: event detection failed ({e}), synthesizing events.")
              ev_raw = None

          if ev_raw is None or len(ev_raw) == 0:
              events_b = synthesize_deap_events_code4(raw_before)
              events_a = synthesize_deap_events_code4(raw_after)
          else:
              events_b = resample_events(ev_raw, raw_src.info["sfreq"], raw_before.info["sfreq"])
              events_a = resample_events(ev_raw, raw_src.info["sfreq"], raw_after.info["sfreq"])

    # keep only code==4
    events_b = events_b[events_b[:,2]==4]
    events_a = events_a[events_a[:,2]==4]

    # Make a lightly filtered BEFORE copy for epoching (so it doesn't drop all)
    raw_before_ep = raw_before.copy()
    raw_before_ep.filter(1., 40., picks="eeg", fir_design="firwin", verbose=False)
    raw_before_ep.notch_filter(50., picks="eeg", verbose=False)

    # Choose epoch params
    if LONG:
        tmin, tmax, base = 0.0, 60.0, None   # full video
    else:
        tmin, tmax, base = -0.2, 3.0, (-0.2, 0.0)

    # Build epochs WITHOUT rejection (long windows)
    epochs_before = mne.Epochs(raw_before_ep, events_b, tmin=tmin, tmax=tmax,
                              baseline=base, picks="eeg", preload=True,
                              reject=None, verbose=False)
    epochs_after  = mne.Epochs(raw_after,    events_a, tmin=tmin, tmax=tmax,
                              baseline=base, picks="eeg", preload=True,
                              reject=None, verbose=False)

    # Ensure same sampling and time grid
    if epochs_before.info["sfreq"] != epochs_after.info["sfreq"]:
        epochs_before.resample(epochs_after.info["sfreq"], npad="auto")

    # Some DEAP files can be a hair short; crop to the common intersection
    common_tmin = max(epochs_before.tmin, epochs_after.tmin)
    common_tmax = min(epochs_before.tmax, epochs_after.tmax)
    epochs_before.crop(tmin=common_tmin, tmax=common_tmax)
    epochs_after.crop(tmin=common_tmin, tmax=common_tmax)

    assert len(epochs_before) > 0 and len(epochs_after) > 0, "No epochs to compare."
    assert len(epochs_before.times) == len(epochs_after.times), "Time grids differ."

    # Average & plot one channel just to visualize (optional for 60 s)
    evk_b = epochs_before.copy().pick(channel).average()
    evk_a = epochs_after.copy().pick(channel).average()
    mne.viz.plot_compare_evokeds({"BEFORE": evk_b, "AFTER": evk_a}, picks=channel, combine=None)
    # ---------- D) Metrics: 50 Hz reduction + ERP SNR ----------
    psd_b, fr  = raw_psd_welch(raw_before, f1, f2)
    psd_a, fr2 = raw_psd_welch(raw_after,  f1, f2)
    assert np.allclose(fr, fr2)
    df = np.diff(fr).mean() if len(fr)>1 else 1.0
    p50_b = (psd_b.sum(axis=1)*df).mean()
    p50_a = (psd_a.sum(axis=1)*df).mean()
    red50 = 100*(1 - p50_a/(p50_b+1e-12))

    def erp_snr(evoked, sig=(0.2,0.5), base=(-0.2,0.0)):
        t = evoked.times; x = evoked.data[0]
        b = (t>=base[0]) & (t<=base[1]); s = (t>=sig[0]) & (t<=sig[1])
        return np.mean(np.abs(x[s]))/(np.std(x[b])+1e-12)

    print(f"50 Hz power reduction: {red50:.1f}%")
    print("ERP SNR (Cz) — BEFORE:", erp_snr(evk_b), "AFTER:", erp_snr(evk_a))


In [ ]:
_ = compare_raw_vs_clean(ROOT, subj="s30", channel="Cz", t_start=120.0, win=5.0)


In [ ]:
from pathlib import Path
import mne

subj = "s24"

# Continuous BEFORE
rb = mne.io.read_raw_fif(str(INTERIM_DIR / f"{subj}-raw.fif"), preload=True, verbose=False)
rb.plot(duration=10, n_channels=32, scalings="auto", picks="eeg")

# Continuous AFTER
ra = mne.io.read_raw_fif(str(INTERIM_DIR / f"{subj}-clean-raw.fif"), preload=True, verbose=False)
ra.plot(duration=10, n_channels=32, scalings="auto", picks="eeg")

# Epoched (AFTER, 3 s)
ep = mne.read_epochs(str(INTERIM_DIR / f"{subj}-epo.fif"), preload=True, verbose=False)
ep.plot(n_epochs=12, scalings="auto", picks="eeg")
ep.average().plot()

# Epoched (AFTER, 60 s)
ep60 = mne.read_epochs(str(INTERIM_DIR / f"{subj}-epo60.fif"), preload=True, verbose=False)
ep60.plot(n_epochs=12, scalings="auto", picks="eeg")
ep60.average().plot()


In [ ]:
subj = "s29"
ep = mne.read_epochs(str(INTERIM_DIR / f"{subj}-epo.fif"), preload=True)

print("Epoch count:", len(ep))
print("Drop log:", ep.drop_log[:10])  # First 10
print("Unique events:", np.unique(ep.events[:,2]))


In [ ]:
import os, numpy as np
from pathlib import Path
import mne
from collections import Counter

ROOT = "/content/drive/MyDrive/Dissertation/Dataset"
INTERIM = f"{ROOT}/data/interim"
epo_path = Path(INTERIM)/f"{subj}-epo60.fif"

print("Epoch file exists? ->", epo_path.exists(), "\n", epo_path)

# Load saved epochs (these are the kept ones)
epochs = mne.read_epochs(str(epo_path), preload=True, verbose=False)
print(epochs)  # shows tmin/tmax, n_channels, sfreq
print("n_epochs kept (saved):", len(epochs))
print("original #trials (length of drop_log):", len(epochs.drop_log))

# Which trials were dropped and why
dropped = [(i, r) for i, r in enumerate(epochs.drop_log) if len(r) > 0]
print("Dropped indices & reasons (first 10):", dropped[:10])

# Reason counts
cnt = Counter()
for rs in epochs.drop_log:
    for r in rs:
        if r not in ("IGNORED",):
            cnt[r] += 1
print("Drop reasons:", dict(cnt))

# Data shape (n_epochs_kept, n_channels, n_times)
print("Data shape:", epochs.get_data().shape)

# If you want the events you used:
print("Events array shape:", epochs.events.shape)   # (n_epochs_kept, 3)


In [ ]:
#Inspecting Epochs

from pathlib import Path
import mne, numpy as np

ROOT = "/content/drive/MyDrive/Dissertation/Dataset"
INTERIM = f"{ROOT}/data/interim"

epochs = mne.read_epochs(str(Path(INTERIM)/f"{subj}-epo60.fif"), preload=True, verbose=False)
print(epochs)  # shows tmin/tmax, n_channels, sfreq
X = epochs.get_data()  # numpy array, shape = (n_epochs, n_channels, n_times)

print("Shape:", X.shape)
print("Kept epochs:", len(epochs))                # e.g., 33
print("Event codes for kept epochs:", np.unique(epochs.events[:,2]))
print("Times (s):", epochs.times[:5], "...", epochs.times[-5:])


In [ ]:
from pathlib import Path
import mne


ROOT = "/content/drive/MyDrive/Dissertation/Dataset"
epo_path = Path(ROOT)/"data"/"interim"/"s29-epo60.fif"

epochs = mne.read_epochs(str(epo_path), preload=False, verbose=False)

print("File:", epo_path)
print("n_epochs (kept):", len(epochs))                 # ← how many epochs in the file
print("n_channels:", len(epochs.ch_names))
print("sfreq (Hz):", epochs.info["sfreq"])
print("tmin..tmax (s):", epochs.tmin, "→", epochs.tmax)
print("epoch window length (s):", epochs.tmax - epochs.tmin)
print("n_times per epoch:", len(epochs.times))
print("events array shape:", epochs.events.shape)      # (n_epochs, 3)
print("original trial indices kept:", epochs.selection.tolist())


In [ ]:
epochs.load_data()                      # loads into RAM
X = epochs.get_data()                   # shape = (n_epochs, n_channels, n_times)
print("Data shape:", X.shape)


In [ ]:
# 3) Verify the new file really is 256 Hz
ep = mne.read_epochs(f"{INTERIM_DIR}/{subj}-epo60.fif", preload=False, verbose=False)
print("sfreq:", ep.info["sfreq"])          # -> 256.0
print("n_times:", len(ep.times))            # -> ~820
print("tmin..tmax:", ep.tmin, "→", ep.tmax)


In [ ]:
# =========================
# FINAL DRIVER CELL (RUN ALL SUBJECTS)
# =========================
from pathlib import Path
import traceback

# --- sanity checks: do not proceed if notebook state is incomplete ---
required = ["CFG", "RAW_DIR", "INTERIM_DIR", "REPORTS_DIR", "SUBJECTS", "load_and_clean_bdf"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(
        f"Missing required objects: {missing}\n"
        "You did not run the setup/function cells. Run the notebook from the top (Runtime -> Run all)."
    )

RAW_DIR = Path(RAW_DIR)
INTERIM_DIR = Path(INTERIM_DIR)
REPORTS_DIR = Path(REPORTS_DIR)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_log = []

for subj in SUBJECTS:
    try:
        # You MUST adapt the next line to match your pipeline:
        # If your function returns raw/epochs/etc, capture what you need.
        # Example assumes you have a single entry-point that does full preprocessing and epoching.
        result = load_and_clean_bdf(subj)  # <-- if signature differs, change ONLY this line

        run_log.append((subj, "OK", ""))

    except Exception as e:
        run_log.append((subj, "FAIL", repr(e)))
        print(f"[FAIL] {subj}: {e}")
        traceback.print_exc()

# Save run log for reporting
import pandas as pd
df_log = pd.DataFrame(run_log, columns=["subject", "status", "error"])
out_csv = REPORTS_DIR / "preprocessing_run_log.csv"
df_log.to_csv(out_csv, index=False)
print("Saved:", out_csv)
df_log


In [ ]:
# --- DIAGNOSTIC: confirm the variables/functions exist ---
needed = ["RAW_DIR", "CFG", "REPORTS_DIR", "load_and_clean_bdf"]
for n in needed:
    print(n, "OK" if n in globals() else "MISSING")

if "load_and_clean_bdf" in globals():
    import inspect
    print("\n--- load_and_clean_bdf signature ---")
    print(inspect.signature(load_and_clean_bdf))
